In [9]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_val_score
from IPython.display import display, Markdown

X_train = pd.read_pickle('../data/processed/tree_ready/X_train.pkl')
X_test = pd.read_pickle('../data/processed/tree_ready/X_test.pkl')
y_train = pd.read_pickle('../data/processed/tree_ready/y_train.pkl')
y_test = pd.read_pickle('../data/processed/tree_ready/y_test.pkl')

results = []

def evaluate(name, model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1] if hasattr(model, 'predict_proba') else None
    r = {'model': name, 'accuracy': accuracy_score(y, pred), 'precision': precision_score(y, pred, zero_division=0),
         'recall': recall_score(y, pred), 'f1': f1_score(y, pred),
         'auc': roc_auc_score(y, proba) if proba is not None else np.nan}
    results.append(r)
    print(r)
    return r

In [10]:
rural_df = pd.read_pickle('../data/interim/rural_ir.pkl')

new_vars = ['m45_1', 's235b', 'h22_1']  # iron tablets, iron-folic acid (15-19), fever last 2 weeks
for v in new_vars:
    print(v, ':', rural_df[v].isnull().mean()*100 if v in rural_df.columns else "NOT FOUND", "% missing")

m45_1 : 79.35833577497803 % missing
s235b : 80.91122179900381 % missing
h22_1 : 69.01552886024027 % missing


In [11]:
ordinal_maps = {
    'v190a': {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},  # poorest -> richest, confirm your actual code order
    'v013': {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6},  # age groups already ordered
}
# Apply explicitly instead of relying on LabelEncoder's arbitrary fit order
for col, mapping in ordinal_maps.items():
    X_train[col] = X_train[col].map(mapping)
    X_test[col] = X_test[col].map(mapping)

In [12]:
def frequency_encode(train_col, test_col):
    freq = train_col.value_counts(normalize=True)
    return train_col.map(freq), test_col.map(freq).fillna(0)

X_train['v113_freq'], X_test['v113_freq'] = frequency_encode(X_train['v113'], X_test['v113'])
X_train['v116_freq'], X_test['v116_freq'] = frequency_encode(X_train['v116'], X_test['v116'])

In [19]:
from catboost import CatBoostClassifier  # pip install catboost

from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer

# CatBoost
cat = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42,
    verbose=0
)

cat.fit(X_train, y_train)

evaluate('CatBoost', cat, X_test, y_test)


# Extra Trees
et = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

et.fit(X_train, y_train)

evaluate('Extra Trees', et, X_test, y_test)


# Gradient Boosting - handle NaN values
imputer = SimpleImputer(strategy='median')

X_train_gb = imputer.fit_transform(X_train)
X_test_gb = imputer.transform(X_test)

gb = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

gb.fit(X_train_gb, y_train)

evaluate('Sklearn Gradient Boosting', gb, X_test_gb, y_test)

{'model': 'CatBoost', 'accuracy': 0.6785185185185185, 'precision': 0.4918032786885246, 'recall': 0.5633802816901409, 'f1': 0.5251641137855579, 'auc': 0.6637857447716602}
{'model': 'Extra Trees', 'accuracy': 0.7022222222222222, 'precision': 0.5275229357798165, 'recall': 0.539906103286385, 'f1': 0.5336426914153132, 'auc': 0.6835507997479828}
{'model': 'Sklearn Gradient Boosting', 'accuracy': 0.7081481481481482, 'precision': 0.5540540540540541, 'recall': 0.38497652582159625, 'f1': 0.45429362880886426, 'auc': 0.6839013881267403}


{'model': 'Sklearn Gradient Boosting',
 'accuracy': 0.7081481481481482,
 'precision': 0.5540540540540541,
 'recall': 0.38497652582159625,
 'f1': 0.45429362880886426,
 'auc': 0.6839013881267403}

In [20]:
from xgboost import XGBClassifier

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                     scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
                     random_state=42, eval_metric='logloss')

auc_scores = cross_val_score(xgb, X_train, y_train, cv=rskf, scoring='roc_auc', n_jobs=-1)
print(f"AUC across 50 folds: mean={auc_scores.mean():.3f}, std={auc_scores.std():.3f}")

AUC across 50 folds: mean=0.679, std=0.018


In [21]:
from sklearn.ensemble import VotingClassifier

with open('../models/ml/random_forest.pkl', 'rb') as f: best_rf = pickle.load(f)
with open('../models/ml/xgboost.pkl', 'rb') as f: best_xgb = pickle.load(f)
with open('../models/ml/lightgbm.pkl', 'rb') as f: best_lgbm = pickle.load(f)

# Weight by individual AUC performance rather than equal voting
voting = VotingClassifier(
    estimators=[('rf', best_rf), ('xgb', best_xgb), ('lgbm', best_lgbm)],
    voting='soft', weights=[1, 1.5, 1.2]  # tune based on each model's standalone AUC
)
voting.fit(X_train, y_train)
evaluate('Weighted Voting Ensemble', voting, X_test, y_test)

[LightGBM] [Info] Number of positive: 851, number of negative: 1845
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000175 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 337
[LightGBM] [Info] Number of data points in the train set: 2696, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

{'model': 'Weighted Voting Ensemble',
 'accuracy': 0.7007407407407408,
 'precision': 0.5248868778280543,
 'recall': 0.5446009389671361,
 'f1': 0.5345622119815668,
 'auc': 0.6844247301993781}

In [22]:
results_df = pd.DataFrame(results).sort_values('auc', ascending=False)
print(results_df)
results_df.to_csv('../results/metrics/accuracy_improvement_experiments.csv', index=False)

                       model  accuracy  precision    recall        f1  \
8   Weighted Voting Ensemble  0.700741   0.524887  0.544601  0.534562   
2  Sklearn Gradient Boosting  0.708148   0.554054  0.384977  0.454294   
7  Sklearn Gradient Boosting  0.708148   0.554054  0.384977  0.454294   
1                Extra Trees  0.702222   0.527523  0.539906  0.533643   
4                Extra Trees  0.702222   0.527523  0.539906  0.533643   
6                Extra Trees  0.702222   0.527523  0.539906  0.533643   
0                   CatBoost  0.678519   0.491803  0.563380  0.525164   
3                   CatBoost  0.678519   0.491803  0.563380  0.525164   
5                   CatBoost  0.678519   0.491803  0.563380  0.525164   

        auc  
8  0.684425  
2  0.683901  
7  0.683901  
1  0.683551  
4  0.683551  
6  0.683551  
0  0.663786  
3  0.663786  
5  0.663786  
